# Guide and environment

This notebook checks the Python environment used to reproduce the study. Read it before choosing the temporal check, model fitting or complete reconstruction. The data describe spectra selected separately for each forest class and date within ottobasin 722936; selected pixels can move between dates.

In [1]:
from pathlib import Path
import os,sys,json
ROOT = next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'config.json').exists())
sys.path.insert(0,str(ROOT))
WORK = Path(os.environ.get('RBC_OUTPUT',ROOT/'work')).resolve()
os.environ['RBC_OUTPUT']=str(WORK)
import pandas as pd
from IPython.display import display,SVG
from dados import CFG


## Choose a reproduction route

The quick route recalculates Seasonal-Trend decomposition using LOESS (STL) from the archived representatives and compares components and metrics with the reference. It needs `rbc-reference.zip`. Figure generation is a separate command, shown in notebook 07.

The model route refits grouped model selection, external evaluation and the operational classifier from archived candidates and human labels. The complete route also reconstructs thematic masks and Minimum Noise Fraction–Pixel Purity Index (MNF–PPI) candidates; it additionally needs `rbc-offline-inputs.zip`.

Run these commands from the extracted code directory, with the required environment active:

```text
python vigente.py quick --work work
python vigente.py model --work work --output-dir rebuilt-model
python vigente.py complete --work work --output-dir rebuilt-complete
```

The first two routes use the scientific Python environment described in `README.md`. Complete MNF–PPI reconstruction requires NumPy 2.1.3 linked to Intel MKL 2023.1, supplied by the native Conda environment. The build check rejects a different linear-algebra backend. Input and code receipts identify executions that can be resumed.

Primary results use seed 13. Adding `--all-seeds` to the quick command also checks supplementary seeds 29 and 47. Their temporal results use seasonal window 7; seed 13 uses all nine windows.

In [2]:
import platform
from importlib.metadata import version
display(pd.Series({m:version(m) for m in ['numpy','pandas','rasterio','scikit-learn','statsmodels']}))
print(platform.platform(),sys.version)
display(CFG)

from threadpoolctl import threadpool_info
display([{k:v for k,v in item.items() if k != "filepath"} for item in threadpool_info()])


numpy            2.1.3
pandas           2.2.3
rasterio         1.4.3
scikit-learn     1.6.1
statsmodels     0.14.4
dtype: object

Windows-11-10.0.26340-SP0 3.13.12 (main, Feb 12 2026, 00:38:53) [MSC v.1944 64 bit (AMD64)]


{'basin': '722936',
 'tiles': ['029010', '030010'],
 'classes': {'1': 'VNFP', '2': 'VNFS', '9': 'Silvicultura'},
 'years': [2016, 2018, 2020, 2022, 2024],
 'catalog': 'data/interim/stac/s2_16d_items.parquet',
 'terra': 'data/raw/terraclass/amazonia',
 'seeds': [13, 29, 47],
 'projections': 10000,
 'min_pixels': 101,
 'pfa': 0.0001,
 'scale': 0.0001,
 'scl_excluded': [0, 1, 2, 3, 8, 9, 10, 11],
 'windows': [7, 9, 11, 13, 15, 21, 25, 31, 35],
 'stl': {'period': 23,
  'seasonal': 7,
  'trend': 45,
  'low_pass': 25,
  'robust': True}}

[{'user_api': 'blas',
  'internal_api': 'openblas',
  'num_threads': 8,
  'prefix': 'libscipy_openblas',
  'version': '0.3.27',
  'threading_layer': 'pthreads',
  'architecture': 'Haswell'},
 {'user_api': 'blas',
  'internal_api': 'openblas',
  'num_threads': 8,
  'prefix': 'libscipy_openblas',
  'version': '0.3.28',
  'threading_layer': 'pthreads',
  'architecture': 'Haswell'}]

## Read the environment output

The environment output lists package versions, calendar parameters and the numerical backends loaded in this kernel. Compare these with `RELEASE_STATUS.json` before starting a route. That report distinguishes model reproduction from full MNF–PPI reconstruction and records the environments actually tested. Human judgments are archived inputs; none of these commands requires another annotation session.

## References

Package versions and installation: `README.md`, `requirements-win.lock`, `requirements-linux.lock`.